In [1]:
## Imports
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from ipywidgets import interact, widgets
from IPython.display import display

plt.style.use("default")
sns.set_theme()

%matplotlib inline

In [2]:
## Find all Weather.csv files

DATA_ROOT = Path("../data/csv")

weather_files = sorted(DATA_ROOT.glob("*/Weather.csv"))

print(f"Found {len(weather_files)} weather files:\n")

for i, f in enumerate(weather_files):
    print(f"{i}: {f.parent.name}")

Found 3 weather files:

0: Huis_19_06_2026_30min
1: Huis_19_06_2026_all
2: Huis_may_2025


In [3]:
## Loader

def load_weather_file(path):
    """
    Load Netatmo-style Weather.csv export.
    """

    df = pd.read_csv(
        path,
        skiprows=5,
        sep=",",
        engine="python"
    )

    df.columns = [str(c).strip() for c in df.columns]

    rename_map = {
        "Timezone : Europe/Amsterdam": "datetime",
    }

    df = df.rename(columns=rename_map)

    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"])

    return df

In [4]:
## Explore a single file

sample_df = load_weather_file(weather_files[0])

print(f"File: {weather_files[0].parent.name}")
print()

display(sample_df.head())

print("\nColumns:")
for col in sample_df.columns:
    print(f" - {col}")

File: Huis_19_06_2026_30min



,Timestamp,datetime,Temperature °C,Humidity %,CO2 ppm,Noise dB,Pressure Pa,Unnamed: 7,Temperature °C.1,Humidity %.1,Unnamed: 10
0,1774999800,2026-04-01 01:30:00,19.8,51,1796,31,1025.4,NaN,8.1,85,NaN
1,1775010600,2026-04-01 04:30:00,19.6,52,1846,31,1024.2,NaN,8.6,85,NaN
2,1775021400,2026-04-01 07:30:00,19.1,53,1381,38,1023.4,NaN,8.4,85,NaN
3,1775032200,2026-04-01 10:30:00,20.9,52,1378,38,1023.0,NaN,10.3,79,NaN
4,1775043000,2026-04-01 13:30:00,22.1,49,1137,39,1021.7,NaN,15.6,61,NaN



Columns:
 - Timestamp
 - datetime
 - Temperature °C
 - Humidity %
 - CO2 ppm
 - Noise dB
 - Pressure Pa
 - Unnamed: 7
 - Temperature °C.1
 - Humidity %.1
 - Unnamed: 10


In [5]:
## Interactive file and variable selector

file_dropdown = widgets.Dropdown(
    options=[(f.parent.name, str(f)) for f in weather_files],
    description="File:",
    layout=widgets.Layout(width="600px")
)

variables_select = widgets.SelectMultiple(
    options=[],
    description="Variables:",
    layout=widgets.Layout(width="500px", height="250px")
)

In [6]:
## Update available variables

def update_columns(change):

    path = Path(change["new"])

    df = load_weather_file(path)

    numeric_columns = [
        col
        for col in df.columns
        if col != "datetime"
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    variables_select.options = numeric_columns

    if numeric_columns:
        variables_select.value = tuple(numeric_columns[:1])


file_dropdown.observe(update_columns, names="value")

update_columns({"new": file_dropdown.value})

In [7]:
## Plot selected variables over time

def plot_weather(file_path, variables):

    if len(variables) == 0:
        print("Select at least one variable.")
        return

    df = load_weather_file(Path(file_path))

    df = df.sort_values("datetime")

    n_vars = len(variables)

    fig, axes = plt.subplots(
        n_vars,
        1,
        figsize=(14, 3 * n_vars),
        sharex=True
    )

    if n_vars == 1:
        axes = [axes]

    for ax, var in zip(axes, variables):

        ax.plot(
            df["datetime"],
            df[var],
            linewidth=1.2
        )

        ax.set_title(var)
        ax.grid(alpha=0.3)

    axes[-1].set_xlabel("Time")

    plt.tight_layout()
    plt.show()

In [8]:
## Interactive weather explorer

interactive_plot = widgets.interactive_output(
    plot_weather,
    {
        "file_path": file_dropdown,
        "variables": variables_select
    }
)

display(
    widgets.VBox([
        file_dropdown,
        variables_select
    ])
)

display(interactive_plot)

Output()